In [2]:
# Import libraries
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

In [3]:
url = "https://books.toscrape.com/"

response = requests.get(url)

print(response.status_code)

200


In [4]:
# Parse the HTML content of the page
soup = BeautifulSoup(response.text, "html.parser")

In [5]:
#FIND all the book titles on the page
books = soup.find_all("article", class_="product_pod")

print(len(books))

20


In [ ]:
#Scraping function to scrape the data from the page
def scrape_page(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    page_data = []

    for book in books:
        title = book.find("h3").find("a")["title"]

        price = book.find(
            "p", class_="price_color"
        ).get_text(strip=True)

        availability = book.find(
            "p", class_="instock availability"
        ).get_text(strip=True)

        rating = book.find(
            "p", class_="star-rating"
        )["class"][1]

        book_url = book.find(
            "h3"
        ).find("a")["href"]

        page_data.append({
            "title": title,
            "price": price,
            "availability": availability,
            "rating": rating,
            "book_url": book_url
        })

    # Find the next page
    next_link = soup.find("li", class_="next")

    if next_link:
        next_url = next_link.find("a")["href"]
    else:
        next_url = None

    return page_data, next_url

In [7]:
all_books = []


In [ ]:
#while loop to scrape all the pages
while url:
    page_data, next_url = scrape_page(url)

    all_books.extend(page_data)

    if next_url:
        url = urljoin(url, next_url)
    else:
        url = None

print("Total books scraped:", len(all_books))

Total books scraped: 1000


In [9]:
df = pd.DataFrame(all_books)

In [10]:
df.head()

,title,price,availability,rating,book_url
0,A Light in the Attic,Â£51.77,In stock,Three,catalogue/a-light-in-the-attic_1000/index.html
1,Tipping the Velvet,Â£53.74,In stock,One,catalogue/tipping-the-velvet_999/index.html
2,Soumission,Â£50.10,In stock,One,catalogue/soumission_998/index.html
3,Sharp Objects,Â£47.82,In stock,Four,catalogue/sharp-objects_997/index.html
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,Five,catalogue/sapiens-a-brief-history-of-humankind...


In [11]:
df.tail()

,title,price,availability,rating,book_url
995,Alice in Wonderland (Alice's Adventures in Won...,Â£55.53,In stock,One,alice-in-wonderland-alices-adventures-in-wonde...
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",Â£57.06,In stock,Four,ajin-demi-human-volume-1-ajin-demi-human-1_4/i...
997,A Spy's Devotion (The Regency Spies of London #1),Â£16.97,In stock,Five,a-spys-devotion-the-regency-spies-of-london-1_...
998,1st to Die (Women's Murder Club #1),Â£53.98,In stock,One,1st-to-die-womens-murder-club-1_2/index.html
999,"1,000 Places to See Before You Die",Â£26.08,In stock,Five,1000-places-to-see-before-you-die_1/index.html


In [12]:
print("Unique book URLs:", df["book_url"].nunique())

Unique book URLs: 1000


In [13]:
df["price"] = (
    df["price"]
    .str.replace("Â£", "", regex=False)
    .astype(float)
)

In [14]:
print(df["price"].head())
print(df["price"].dtype)

0    51.77
1    53.74
2    50.10
3    47.82
4    54.23
Name: price, dtype: float64
float64


In [15]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["rating"].map(rating_map)

In [16]:
print(df["rating"].value_counts().sort_index())

rating
1    226
2    196
3    203
4    179
5    196
Name: count, dtype: int64


In [17]:
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

Dataset shape: (1000, 5)

Missing values:
title           0
price           0
availability    0
rating          0
book_url        0
dtype: int64

Duplicate rows: 0

Data types:
title               str
price           float64
availability        str
rating            int64
book_url            str
dtype: object


In [19]:
df.head()

,title,price,availability,rating,book_url
0,A Light in the Attic,51.77,In stock,3,catalogue/a-light-in-the-attic_1000/index.html
1,Tipping the Velvet,53.74,In stock,1,catalogue/tipping-the-velvet_999/index.html
2,Soumission,50.10,In stock,1,catalogue/soumission_998/index.html
3,Sharp Objects,47.82,In stock,4,catalogue/sharp-objects_997/index.html
4,Sapiens: A Brief History of Humankind,54.23,In stock,5,catalogue/sapiens-a-brief-history-of-humankind...


In [18]:
df.to_csv("../data/books_cleaned.csv", index=False)

Business Value

This scraper demonstrates how publicly available e-commerce catalogue data can be collected and transformed into a structured dataset for analysis. The resulting dataset can support use cases such as price monitoring, product catalogue analysis, rating distribution analysis, and competitive intelligence. In a real-world implementation, the same pipeline could be scheduled to run periodically to monitor changes in product prices and availability.

Limitation

The project uses a sandbox website designed specifically for web-scraping practice. Therefore, the dataset does not represent a live commercial marketplace. The scraper also collects catalogue-level information only and does not attempt to bypass authentication, access restricted content, or circumvent anti-bot mechanisms.